# MX017 Shadow diagnosis — 4 October 2026
Read-only companion to the saved Candidate audit. Reference precedes Shadow. This notebook reads recorded outputs only; it does not refit, activate, or modify database state. Raw slices preserve all source states and missing seconds.

In [1]:
from pathlib import Path
import json, pandas as pd
root=Path.cwd()
out=root/'notebooks/outputs/mx017_shadow_diagnosis_20261004'
r=json.loads((out/'receipt.json').read_text(encoding='utf-8'))
assert all(m['source_digest_match'] for m in r['modules'])
print('Parent:', r['parent_id'])
print('Created UTC:', r['parent_created_utc'])
print('All eight frozen source digests match:', True)

Parent: ee60a6b2-8fe0-4301-9bbb-3c0249373697
Created UTC: 2026-10-04 14:30:05.797185
All eight frozen source digests match: True


In [2]:
rows=[]
for m in r['modules']:
 a=m.get('threshold_comparator_audit',{})
 rows.append({'module':m['module'],'shadow_minutes':m['summary']['shadow_minutes'],'current_pct':m['summary']['shadow_flag_pct'],'strict_gt_pct_diagnostic_only':a.get('shadow_gt_pct'),'valve_only_minutes':m.get('valve_only_minutes'),'quarantine_pct':m['summary']['quarantine_pct']})
display(pd.DataFrame(rows))

,module,shadow_minutes,current_pct,strict_gt_pct_diagnostic_only,valve_only_minutes,quarantine_pct
0,1,4262,34.209291,0.539653,1435.0,3.261790
1,2,4263,34.248182,0.469153,1440.0,11.044347
2,3,4263,0.117288,0.117288,0.0,8.024256
3,4,4263,4.410040,4.410040,0.0,1.535162
4,5,4261,2.041774,2.041774,0.0,8.638315
5,6,4261,0.328561,0.328561,0.0,7.900607
6,7,0,NaN,NaN,NaN,0.000000
7,8,4264,0.914634,0.914634,0.0,0.617946


In [3]:
for m in r['modules'][:2]:
 print('M',m['module'],'Valve baseline:',m['baseline']['valve'],'threshold:',m['threshold'])
 print(m['threshold_comparator_audit'])
 print('Episodes:',m['episode_summary'])

M 1 Valve baseline: {'median': 102.0, 'scale': 1.0} threshold: 5.0
{'comparison': 'Diagnostic only; runtime remains >=; threshold unchanged', 'reference_retained_ge': 529, 'reference_retained_gt': 144, 'reference_retained_equal': 385, 'reference_retained_minutes': 34255, 'shadow_ge': 1458, 'shadow_gt': 23, 'shadow_equal': 1435, 'shadow_minutes': 4262, 'shadow_gt_pct': 0.5396527451900516}
Episodes: {'count': 19, 'flagged_minutes': 1458, 'single_minute_runs': 2, 'runs_le_5min': 2, 'minutes_in_runs_le_5min': 2, 'runs_ge_30min': 12, 'minutes_in_runs_ge_30min': 1406, 'longest_minutes': 390, 'median_minutes': 46.0, 'returned_to_below_threshold': 2, 'ended_at_state_or_gap': 17}
M 2 Valve baseline: {'median': 92.0, 'scale': 1.0} threshold: 8.0
{'comparison': 'Diagnostic only; runtime remains >=; threshold unchanged', 'reference_retained_ge': 401, 'reference_retained_gt': 27, 'reference_retained_equal': 374, 'reference_retained_minutes': 31533, 'shadow_ge': 1460, 'shadow_gt': 20, 'shadow_equal'

In [4]:
raw=json.loads((out/'raw_checks.json').read_text(encoding='utf-8'))
for s in raw:
 print('M',s['module'],s['start'],s['end'],'seconds:',s['observed_rows'],'/',s['expected_seconds'],'SV:',s['sv_counts'])
 display(pd.DataFrame(s['minute_means'][::5]))

M 1 2026-09-29T23:30:00 2026-09-30T00:00:00 seconds: 1775 / 1800 SV: {'On': 289, 'Off': 1486}


,time,observed_seconds,sv,hp2,lp2,temphi,templo,valve
0,2026-09-29T23:30,59,{'On': 59},316.694915,52.118644,106.289831,10.762712,97.0
1,2026-09-29T23:35,59,{'Off': 59},311.450847,49.200000,110.889831,10.050847,97.0
2,2026-09-29T23:40,59,{'Off': 59},205.118644,22.506780,98.277966,-58.533898,97.0
3,2026-09-29T23:45,60,{'Off': 60},195.770000,17.656667,79.180000,-64.828333,97.0
4,2026-09-29T23:50,60,{'Off': 60},194.126667,16.798333,73.961667,-65.606667,97.0
5,2026-09-29T23:55,59,{'Off': 59},194.116949,16.523729,72.276271,-65.896610,97.0


M 1 2026-09-30T00:10:00 2026-09-30T00:20:00 seconds: 591 / 600 SV: {'Off': 591}


,time,observed_seconds,sv,hp2,lp2,temphi,templo,valve
0,2026-09-30T00:10,59,{'Off': 59},198.462712,17.230508,67.030508,-65.428814,97.0
1,2026-09-30T00:15,59,{'Off': 59},195.411864,17.076271,66.267797,-65.525424,97.0


M 4 2026-09-30T01:50:00 2026-09-30T02:20:00 seconds: 1775 / 1800 SV: {'Off': 1775}


,time,observed_seconds,sv,hp2,lp2,temphi,templo,valve
0,2026-09-30T01:50,60,{'Off': 60},199.601667,15.225000,79.001667,-65.118333,83.0
1,2026-09-30T01:55,59,{'Off': 59},198.950847,14.006780,82.764407,-52.230508,83.0
2,2026-09-30T02:00,59,{'Off': 59},198.681356,14.067797,84.622034,-52.662712,83.0
3,2026-09-30T02:05,59,{'Off': 59},199.803390,14.179661,85.269492,-52.949153,83.0
4,2026-09-30T02:10,59,{'Off': 59},198.088136,14.164407,85.525424,-52.803390,83.0
5,2026-09-30T02:15,59,{'Off': 59},198.283051,14.174576,85.894915,-53.301695,83.0


In [5]:
d=pd.read_csv(out/'distributions.csv')
display(d[(d.module==1)&d.feature.isin(['hp2','lp2','temphi','templo','valve'])&d.phase.isin(['reference_retained','shadow'])])

,module,feature,phase,minutes,p01,p10,median,p90,p99
7,1,hp2,reference_retained,34255,191.832814,193.879661,196.201695,200.506734,203.956441
8,1,hp2,shadow,4262,192.804136,194.880333,198.488234,199.342373,202.977254
10,1,lp2,reference_retained,34255,16.610169,17.135593,18.264407,18.984746,19.765186
11,1,lp2,shadow,4262,15.930508,16.759322,17.033898,17.379661,17.864068
19,1,valve,reference_retained,34255,97.000000,101.000000,102.000000,102.000000,102.000000
20,1,valve,shadow,4262,97.000000,97.000000,102.000000,102.000000,102.000000
22,1,temphi,reference_retained,34255,58.357627,59.262712,61.001695,64.894915,69.879661
23,1,temphi,shadow,4262,56.047458,59.445763,60.062189,64.931500,67.584661
25,1,templo,reference_retained,34255,-66.057627,-65.700000,-64.954237,-64.569492,-64.274576
26,1,templo,shadow,4262,-66.505033,-66.009492,-65.638983,-65.464407,-65.098818


## Interpretation limits
Threshold ties explain most M1/M2 raw-flag burden. Genuine pressure/thermal transitions and sustained M4/M5 deviations remain separate review cases. Stable is not healthy. Neither a strict-comparison counterfactual nor a single settling trace authorizes Active promotion. Proposed next work: versioned discrete-feature boundary handling, control context versus physical evidence, independently validated phase-aware envelopes, and readable per-check blockers.